# 33 — Calibración, ranking y congelamiento de selección

Divide temporalmente VALIDATION en dos partes: una para ajustar calibración y otra posterior para comparar modelos/calibradores. Al final congela `selected_model.json` **antes de cualquier acceso a TEST**.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "32_validation.parquet").exists(), "Ejecuta 32 primero."
from argos.models.calibration import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


In [ ]:
from argos.models.calibration import build_calibration_and_selection

result = build_calibration_and_selection(
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
    modeling_spec=MODEL_SPEC,
)
result.candidate_metrics


In [ ]:
result.calibration_metrics

In [ ]:
result.ranking

In [ ]:
result.reliability

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("33 ✅ PASS" if result.ready_for_next_notebook else "33 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Frozen selection:", result.selection_path)
assert result.ready_for_next_notebook
